# Futures/Spot HBT Step-by-Step Notebook

This notebook calls functions from `future_spot/scripts/run_hbt_daily_full_market_backtest.py` directly, one stage per cell. It keeps the CLI pipeline visible while ending with summary tables, charts, and three-timeline latency observation.


In [ ]:
from argparse import Namespace
from pathlib import Path
import logging
import sys

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.4f}".format)

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SCRIPT_DIR = PROJECT_ROOT / "scripts"

for path in (SCRIPT_DIR, PROJECT_ROOT, WORKSPACE_ROOT):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

import run_hbt_daily_full_market_backtest as daily_hbt

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print(f"project_root={PROJECT_ROOT}")
print(f"workspace_root={WORKSPACE_ROOT}")


## 1. Parameters

Keep these paths Linux/WSL friendly. Windows-style paths such as `Z:\...` will not open correctly from Linux Python.

This notebook is configured for latency observation with `order_latency_ms=5.0`, `response_latency_ms=5.0`, and no extra feed latency offset.


In [ ]:
args = Namespace(
    start_date="2026-05-21",
    end_date="2026-05-26",
    base_config=PROJECT_ROOT / "arbitrage_config_20260702.json",
    calendar=PROJECT_ROOT / "Calendar.csv",
    stockinfo=PROJECT_ROOT / "stockinfo.csv",
    output_dir=PROJECT_ROOT / "output" / "hbt_daily_full_market_20260521_20260526_latency_5ms",

    futures_parquet_template="/mnt/z/ticks_parquet_stock_future/{ldate}.parquet",
    twse_daytrade_template="/mnt/z/TWSE/每日個股狀況/{date_nodash}.csv",
    tpex_daytrade_template="/mnt/z/TPEX/每日個股狀況/{date_nodash}.csv",
    twse_daily_template="/mnt/z/TWSE/每日資料/{ldate_nodash}.ftr",
    tpex_daily_template="/mnt/z/TPEX/每日資料/{ldate_nodash}.ftr",
    data_platform_base="/mnt/z/數據平台",
    event_futures_parquet_dir=Path("/mnt/z/ticks_parquet_stock_future"),

    build_session_start="08:45:00",
    build_session_end="13:45:00",
    min_future_volume=1000,
    min_stock_volume=20_000_000,
    required_unit=2000,
    name_template="{spot_symbol}_{future_symbol}",
    rebuild_daily_configs=False,

    session_start="09:00:00",
    session_end="13:30:00",
    pair_name=[],
    max_pairs=None,

    no_convert_missing_event_data=False,
    rebuild_event_data=False,
    conversion_qa_sample_rows=1000,

    first_leg="future",
    step_ms=1000.0,
    order_latency_ms=5.0,
    response_latency_ms=5.0,
    feed_latency_offset_ms=0.0,
    second_leg_delay_ms=0.0,
    response_timeout_ms=50.0,
    max_steps=None,
    max_trades_per_pair=None,
    record_market_every_steps=1,
    queue_model="risk_adverse",
    entry_threshold_pct=None,
    exit_threshold_pct=None,
    min_effective_tick_multiple=None,
    min_second_leg_adjusted_basis_pct=None,
    no_second_leg_profit_check=False,
    no_flatten=False,

    continue_on_error=False,
    log_level="INFO",
)

args.output_dir.mkdir(parents=True, exist_ok=True)
print(args.output_dir)


## 2. Select Trade Dates

Calls `select_trade_dates()` from the runner.


In [ ]:
trade_dates = daily_hbt.select_trade_dates(args.calendar, args.start_date, args.end_date)
trade_dates


## 3. Build Daily Pair Universe

Calls `build_daily_pair_records()`, then writes the same CSVs as the CLI.


In [ ]:
records, build_status = daily_hbt.build_daily_pair_records(args, trade_dates)
daily_hbt.write_csv(build_status, args.output_dir / "daily_config_build_status.csv")

pair_universe = daily_hbt.pair_universe_frame(records)
daily_hbt.write_csv(pair_universe, args.output_dir / "daily_pair_universe.csv")

print(f"records={len(records):,}")
build_status


In [ ]:
pair_universe.head(20)

## 4. Build / Reuse HBT Event Data

Calls `build_event_data()`. With `rebuild_event_data=False`, existing `.npz` files are reused.


In [ ]:
event_paths, conversion_status = daily_hbt.build_event_data(args, records)
daily_hbt.write_csv(conversion_status, args.output_dir / "conversion_status.csv")

print(f"ready_pairs={len(event_paths):,}")
conversion_status.value_counts(["spot_status", "future_status", "ok"], dropna=False).reset_index(name="rows")


## 5. HBT Settings Audit

Calls `hbt_settings_frame()` to inspect tick sizes, event rows, and basic event counts before running HBT.


In [ ]:
settings = daily_hbt.hbt_settings_frame(args, records, event_paths)
daily_hbt.write_csv(settings, args.output_dir / "hbt_settings.csv")

settings.groupby("leg").agg(
    assets=("symbol", "count"),
    rows=("rows", "sum"),
    depth_events=("depth_events", "sum"),
    trade_events=("trade_events", "sum"),
)


## 6. Run HBT Backtests

Calls `run_backtests()`. This is the expensive step. It returns in-memory DataFrames and then writes the same output CSVs as the CLI, including `latency_all_daily_pairs.csv`.


In [ ]:
pair_results, summary, trades, market, latency, run_errors = daily_hbt.run_backtests(args, records, event_paths)

daily_hbt.write_csv(summary, args.output_dir / "summary_all_daily_pairs.csv")
daily_hbt.write_csv(trades, args.output_dir / "trades_all_daily_pairs.csv")
daily_hbt.write_csv(market, args.output_dir / "market_all_daily_pairs.csv")
daily_hbt.write_csv(latency, args.output_dir / "latency_all_daily_pairs.csv")
daily_hbt.write_csv(run_errors, args.output_dir / "run_errors.csv")

print(f"completed_pairs={len(pair_results):,} latency_rows={len(latency):,} errors={len(run_errors):,}")
summary.head()


## 7. Build Entry / Exit Tables

Calls `build_entry_exit_outputs()` and writes compact per-pair entry / execution files.


In [ ]:
entry_exit_by_pair, entry_exit_all, entry_exit_index = daily_hbt.build_entry_exit_outputs(pair_results, records)

daily_hbt.write_csv(entry_exit_all, args.output_dir / "entry_exit_all_daily_pairs.csv")
daily_hbt.write_csv(entry_exit_index, args.output_dir / "entry_exit_index.csv")
daily_hbt.write_entry_exit_by_pair(entry_exit_by_pair, args.output_dir / "entry_exit_by_pair")

entry_exit_index.head()


## 8. Entry / Exit Rules

**Long spot / short future entry**

- `long_spot_short_future_pct >= entry_threshold_pct`.
- `long_spot_short_future_ticks > min_effective_tick_multiple`.
- `spot_ask_size >= stock_min_ask_size`.
- `future_bid_size >= future_min_bid_size`.

**Execution / exit**

- Default `first_leg` is `future`.
- Second leg is checked after first-leg fill.
- Existing positions exit when the configured exit tick or reverse-basis rule is triggered.
- Final estimated profit is `realized_pnl`.


In [ ]:
rule_summary = (
    pair_universe
    .groupby(["entry_threshold_pct", "min_effective_tick_multiple", "spot_order_qty", "future_order_qty", "future_pnl_multiplier"], dropna=False)
    .size()
    .reset_index(name="daily_pair_count")
    .sort_values("daily_pair_count", ascending=False)
)
rule_summary


## 9. Estimated Profit by Symbol / Pair


In [ ]:
symbol_profit = (
    summary
    .groupby("spot_symbol", as_index=False)
    .agg(
        trade_days=("trade_date", "nunique"),
        pair_names=("pair_name", "nunique"),
        estimated_profit=("realized_pnl", "sum"),
        avg_daily_profit=("realized_pnl", "mean"),
        filled_pairs=("filled_pairs", "sum"),
        second_leg_failures=("second_leg_failures", "sum"),
        flatten_count=("flatten_count", "sum"),
        final_quantity_abs=("final_quantity", lambda s: s.abs().sum()),
    )
    .sort_values("estimated_profit", ascending=False)
)

symbol_profit.head(30)


In [ ]:
pair_profit = summary.merge(
    pair_universe[["run_key", "entry_threshold_pct", "min_effective_tick_multiple", "spot_order_qty", "future_order_qty"]],
    on="run_key",
    how="left",
).rename(columns={"realized_pnl": "estimated_profit"})

pair_profit = pair_profit[[
    "trade_date", "pair_name", "spot_symbol", "future_symbol",
    "estimated_profit", "filled_pairs", "second_leg_failures", "flatten_count",
    "final_quantity", "entry_threshold_pct", "min_effective_tick_multiple",
    "spot_order_qty", "future_order_qty",
]].sort_values("estimated_profit", ascending=False)

pair_profit.head(50)


## 10. Stuck Cash Flow

Estimate stock-side cash still tied up after arbitrage entries by subtracting convergence-exit cash recovered from entry cash deployed. This is a stock cash view; futures margin and futures PnL are separate constraints.


In [ ]:
pair_cash_settings = pd.DataFrame([
    {
        "run_key": record.run_key,
        "spot_order_qty": record.pair.spot_order_qty,
        "stock_commission_rate": record.pair.stock_commission_rate,
        "stock_commission_discount": record.pair.stock_commission_discount,
        "stock_transaction_tax_rate": record.pair.stock_transaction_tax_rate,
    }
    for record in records
])

filled_trades = trades.loc[trades["status"].eq("FILLED")].copy()
for column in ["first_exec_price", "second_exec_price"]:
    filled_trades[column] = pd.to_numeric(filled_trades[column], errors="coerce")

filled_trades["spot_exec_price"] = filled_trades["second_exec_price"].where(
    filled_trades["second_leg"].eq("stock"),
    filled_trades["first_exec_price"],
)
filled_trades["spot_side"] = filled_trades["second_side"].where(
    filled_trades["second_leg"].eq("stock"),
    filled_trades["first_side"],
)

cash_trades = filled_trades.merge(pair_cash_settings, on="run_key", how="left")
cash_trades["stock_notional"] = cash_trades["spot_exec_price"] * cash_trades["spot_order_qty"]
cash_trades["commission_rate"] = cash_trades["stock_commission_rate"] * cash_trades["stock_commission_discount"]
cash_trades["stock_fee"] = cash_trades["stock_notional"] * cash_trades["commission_rate"]
cash_trades["stock_tax"] = cash_trades["stock_notional"] * cash_trades["stock_transaction_tax_rate"]

entry_cash = cash_trades.loc[cash_trades["signal"].eq("ENTER_LONG_SPOT_SHORT_FUTURE")].copy()
exit_cash = cash_trades.loc[cash_trades["signal"].eq("EXIT")].copy()

entry_cash["entry_cash_out"] = entry_cash["stock_notional"] + entry_cash["stock_fee"]
exit_cash["exit_cash_in"] = exit_cash["stock_notional"] - exit_cash["stock_fee"] - exit_cash["stock_tax"]

entry_by_pair = entry_cash.groupby(
    ["trade_date", "run_key", "pair_name", "spot_symbol", "future_symbol"],
    as_index=False,
).agg(
    entries=("signal", "size"),
    entry_cash_out=("entry_cash_out", "sum"),
    entry_stock_notional=("stock_notional", "sum"),
    avg_entry_spot=("spot_exec_price", "mean"),
)

exit_by_pair = exit_cash.groupby(
    ["trade_date", "run_key", "pair_name"],
    as_index=False,
).agg(
    exits=("signal", "size"),
    exit_cash_in=("exit_cash_in", "sum"),
    exit_stock_notional=("stock_notional", "sum"),
    avg_exit_spot=("spot_exec_price", "mean"),
)

stuck_cash_by_pair = entry_by_pair.merge(
    exit_by_pair,
    on=["trade_date", "run_key", "pair_name"],
    how="left",
).fillna({
    "exits": 0,
    "exit_cash_in": 0.0,
    "exit_stock_notional": 0.0,
})
stuck_cash_by_pair["open_pairs"] = stuck_cash_by_pair["entries"] - stuck_cash_by_pair["exits"]
stuck_cash_by_pair["net_stock_cash_stuck"] = stuck_cash_by_pair["entry_cash_out"] - stuck_cash_by_pair["exit_cash_in"]

stuck_cash_summary = pd.DataFrame([
    {
        "entry_rows": len(entry_cash),
        "exit_rows": len(exit_cash),
        "open_pairs": stuck_cash_by_pair["open_pairs"].sum(),
        "entry_cash_out": entry_cash["entry_cash_out"].sum(),
        "exit_cash_in": exit_cash["exit_cash_in"].sum(),
        "net_stock_cash_stuck": stuck_cash_by_pair["net_stock_cash_stuck"].sum(),
    }
])

stuck_cash_summary


In [ ]:
daily_stuck_cash = (
    stuck_cash_by_pair
    .groupby("trade_date", as_index=False)
    .agg(
        entries=("entries", "sum"),
        exits=("exits", "sum"),
        open_pairs=("open_pairs", "sum"),
        entry_cash_out=("entry_cash_out", "sum"),
        exit_cash_in=("exit_cash_in", "sum"),
        net_stock_cash_stuck=("net_stock_cash_stuck", "sum"),
    )
)

daily_stuck_cash


In [ ]:
top_stuck_cash_pairs = stuck_cash_by_pair.sort_values(
    "net_stock_cash_stuck",
    ascending=False,
)[[
    "trade_date", "pair_name", "spot_symbol", "future_symbol",
    "entries", "exits", "open_pairs",
    "entry_cash_out", "exit_cash_in", "net_stock_cash_stuck",
    "avg_entry_spot", "avg_exit_spot",
]]

top_stuck_cash_pairs.head(30)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_daily_cash = daily_stuck_cash.copy()
plot_daily_cash["date_label"] = pd.to_datetime(plot_daily_cash["trade_date"]).dt.strftime("%m-%d")
axes[0].bar(plot_daily_cash["date_label"], plot_daily_cash["net_stock_cash_stuck"], color="#0f766e")
axes[0].set_title("Daily Net Stock Cash Stuck")
axes[0].set_xlabel("trade date")
axes[0].set_ylabel("cash")

plot_top_cash = top_stuck_cash_pairs.head(15).sort_values("net_stock_cash_stuck")
axes[1].barh(plot_top_cash["pair_name"], plot_top_cash["net_stock_cash_stuck"], color="#b45309")
axes[1].set_title("Top 15 Pairs by Stuck Stock Cash")
axes[1].set_xlabel("cash")

fig.tight_layout()
plt.show()


## 11. Latency Observation

The long-format `latency` table records the local timeline plus the latest spot/future exchange feed timestamps and order timestamps at each lifecycle event.


In [ ]:
latency_summary = (
    latency
    .groupby(["pair_name", "event_type"], dropna=False)
    .agg(
        rows=("event_type", "size"),
        spot_feed_latency_ms=("spot_feed_latency_ns", lambda s: pd.to_numeric(s, errors="coerce").mean() / 1_000_000),
        future_feed_latency_ms=("future_feed_latency_ns", lambda s: pd.to_numeric(s, errors="coerce").mean() / 1_000_000),
        order_entry_latency_ms=("order_entry_latency_ns", lambda s: pd.to_numeric(s, errors="coerce").mean() / 1_000_000),
        order_response_latency_ms=("order_response_latency_ns", lambda s: pd.to_numeric(s, errors="coerce").mean() / 1_000_000),
    )
    .reset_index()
)

latency_summary.sort_values(["pair_name", "event_type"]).head(60)


In [ ]:
latency_event_counts = latency.value_counts(["event_type", "leg", "side"], dropna=False).reset_index(name="rows")
latency_event_counts


In [ ]:
order_latency_check = latency.loc[latency["order_entry_latency_ns"].notna()].copy()
order_latency_check[["event_type", "leg", "side", "order_entry_latency_ns", "order_response_latency_ns"]].describe(include="all")


## 12. Visualization


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
summary_plot = summary.copy()
summary_plot["trade_date"] = pd.to_datetime(summary_plot["trade_date"])

daily = summary_plot.groupby("trade_date", as_index=False).agg(
    estimated_profit=("realized_pnl", "sum"),
    filled_pairs=("filled_pairs", "sum"),
    second_leg_failures=("second_leg_failures", "sum"),
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(daily["trade_date"].dt.strftime("%m-%d"), daily["estimated_profit"], color="#2563eb")
axes[0, 0].axhline(0, color="#111827", linewidth=0.8)
axes[0, 0].set_title("Daily Estimated Profit")
axes[0, 0].set_ylabel("realized_pnl")

top = symbol_profit.head(15).sort_values("estimated_profit")
axes[0, 1].barh(top["spot_symbol"].astype(str), top["estimated_profit"], color="#059669")
axes[0, 1].set_title("Top 15 Symbols by Estimated Profit")
axes[0, 1].set_xlabel("realized_pnl")

bottom = symbol_profit.tail(15).sort_values("estimated_profit")
axes[1, 0].barh(bottom["spot_symbol"].astype(str), bottom["estimated_profit"], color="#dc2626")
axes[1, 0].axvline(0, color="#111827", linewidth=0.8)
axes[1, 0].set_title("Bottom 15 Symbols by Estimated Profit")
axes[1, 0].set_xlabel("realized_pnl")

axes[1, 1].plot(daily["trade_date"].dt.strftime("%m-%d"), daily["filled_pairs"], marker="o", color="#7c3aed", label="filled")
axes[1, 1].plot(daily["trade_date"].dt.strftime("%m-%d"), daily["second_leg_failures"], marker="o", color="#f97316", label="second-leg failures")
axes[1, 1].set_title("Daily Execution Count")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()

fig.tight_layout()
plt.show()


In [ ]:
plot_df = pair_profit.copy()
plot_df["date_label"] = pd.to_datetime(plot_df["trade_date"]).dt.strftime("%m-%d")

fig, ax = plt.subplots(figsize=(16, 5))
for date_label, part in plot_df.groupby("date_label"):
    ax.scatter(part["filled_pairs"], part["estimated_profit"], s=32, alpha=0.65, label=date_label)
ax.axhline(0, color="#111827", linewidth=0.8)
ax.set_title("Pair Estimated Profit vs Filled Pairs")
ax.set_xlabel("filled_pairs")
ax.set_ylabel("estimated_profit")
ax.legend(title="trade date", ncols=4)
plt.show()


## 13. Selected Pair Latency Drill-Down


In [ ]:
SELECTED_PAIR = pair_profit.iloc[0]["pair_name"]

entry_cols = [
    "trade_date", "time", "pair_name", "entry_signal", "row_type", "status",
    "long_spot_short_future_pct", "long_spot_short_future_ticks",
    "short_spot_long_future_pct", "short_spot_long_future_ticks",
    "first_leg", "first_side", "first_exec_price", "first_exec_qty",
    "second_leg", "second_side", "second_exec_price", "second_exec_qty",
    "realized_pnl", "first_to_second_exch_ms",
]

entry_exit_all.loc[entry_exit_all["pair_name"].eq(SELECTED_PAIR), [c for c in entry_cols if c in entry_exit_all.columns]].head(100)


In [ ]:
latency_view = latency.loc[latency["pair_name"].eq(SELECTED_PAIR)].copy()
latency_view["local_time"] = pd.to_datetime(latency_view["local_ts"], unit="ns", utc=True).dt.tz_convert("Asia/Taipei")
latency_view["spot_exch_time"] = pd.to_datetime(latency_view["spot_exch_ts"], unit="ns", utc=True).dt.tz_convert("Asia/Taipei")
latency_view["future_exch_time"] = pd.to_datetime(latency_view["future_exch_ts"], unit="ns", utc=True).dt.tz_convert("Asia/Taipei")

cols = [
    "trade_date", "pair_name", "event_type", "leg", "side", "status",
    "local_time", "spot_exch_time", "future_exch_time",
    "spot_feed_latency_ns", "future_feed_latency_ns",
    "order_req_local_ts", "order_exch_ts", "order_resp_local_ts",
    "order_entry_latency_ns", "order_response_latency_ns",
]
latency_view[[c for c in cols if c in latency_view.columns]].head(120)


In [ ]:
plot_latency = latency_view.head(80).copy().reset_index(drop=True)
if plot_latency.empty:
    print(f"No latency rows for {SELECTED_PAIR}")
else:
    x = plot_latency.index
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(x, plot_latency["local_ts"] / 1_000_000, marker="o", label="local")
    ax.plot(x, plot_latency["spot_exch_ts"] / 1_000_000, marker="o", label="spot_exch")
    ax.plot(x, plot_latency["future_exch_ts"] / 1_000_000, marker="o", label="future_exch")
    ax.set_title(f"Three Timelines: {SELECTED_PAIR}")
    ax.set_xlabel("latency event index")
    ax.set_ylabel("timestamp ms")
    ax.legend()
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(x)
    ax2.set_xticklabels(plot_latency["event_type"], rotation=60, ha="left", fontsize=8)
    fig.tight_layout()
    plt.show()
